# NEXUS Quantum Wave Distillation on Colab

Run this notebook top-to-bottom.

What it does:
- clones or pulls the repo
- installs a dedicated `xtb` runtime in a separate Miniforge env
- runs `scripts/distill_quantum_wave_targets.py` on an SDF dataset
- saves a `.pt` payload keyed by canonical SMILES for later Phase 2 training


In [ ]:
# Cell 1: clone/pull repo
import os, subprocess

REPO = 'https://github.com/Nikku03/enzyme_Software.git'
REPO_DIR = '/content/enzyme_Software'

if not os.path.exists(REPO_DIR):
    print('Cloning repo...')
    subprocess.run(f'git clone {REPO} {REPO_DIR}', shell=True, check=True)
else:
    print('Repo exists, pulling latest...')
    subprocess.run(f'git -C {REPO_DIR} pull origin main', shell=True, check=True)

subprocess.run(f'git -C {REPO_DIR} log --oneline -5', shell=True, check=True)


In [ ]:
# Cell 2: optional Google Drive mount for persistent outputs
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted.')
except Exception as e:
    print('Drive mount skipped:', e)


In [ ]:
# Cell 3: configure distillation targets
import os

SDF_TARGET = '/content/enzyme_Software/data/ATTNSOM/cyp_dataset/3A4.sdf'
OUTPUT_TARGET = '/content/drive/MyDrive/nexus_3a4_quantum_features.pt'
CACHE_DIR = '/content/drive/MyDrive/cache/quantum_wave_xtb'
MINIFORGE_DIR = '/content/miniforge3'
ENV_NAME = 'xtb-env'

print('SDF_TARGET   =', SDF_TARGET)
print('OUTPUT_TARGET=', OUTPUT_TARGET)
print('CACHE_DIR    =', CACHE_DIR)


In [ ]:
# Cell 4: install Miniforge + xtb + Python dependencies used by the distiller
import os, subprocess, textwrap

if not os.path.exists(MINIFORGE_DIR):
    subprocess.run(
        'wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/miniforge.sh',
        shell=True,
        check=True,
    )
    subprocess.run(f'bash /tmp/miniforge.sh -b -p {MINIFORGE_DIR}', shell=True, check=True)

install_script = textwrap.dedent(f'''\
source {MINIFORGE_DIR}/etc/profile.d/conda.sh
if ! conda env list | awk '{{print $1}}' | grep -qx {ENV_NAME}; then
  conda create -y -n {ENV_NAME} -c conda-forge python=3.12 xtb numpy pytorch tqdm rdkit
else
  conda install -y -n {ENV_NAME} -c conda-forge xtb numpy pytorch tqdm rdkit
fi
conda run -n {ENV_NAME} xtb --version
conda run -n {ENV_NAME} python - <<'PY'
import numpy, torch, tqdm
from rdkit import Chem
print('numpy', numpy.__version__)
print('torch', torch.__version__)
print('rdkit', Chem.rdBase.rdkitVersion)
PY
''')
subprocess.run(['bash', '-lc', install_script], check=True)


In [ ]:
# Cell 5: run the distiller
import subprocess, textwrap

run_script = textwrap.dedent(f'''\
source {MINIFORGE_DIR}/etc/profile.d/conda.sh
conda run -n {ENV_NAME} python /content/enzyme_Software/scripts/distill_quantum_wave_targets.py \
  --sdf "{SDF_TARGET}" \
  --output "{OUTPUT_TARGET}" \
  --cache-dir "{CACHE_DIR}"
''')
proc = subprocess.run(
    ['bash', '-lc', run_script],
    text=True,
    capture_output=True,
)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError(f'distiller failed with exit code {proc.returncode}')


In [ ]:
# Cell 6: inspect the saved payload
import torch

payload = torch.load(OUTPUT_TARGET, map_location='cpu', weights_only=False)
print('molecules:', len(payload))
first_key = next(iter(payload))
print('first smiles:', first_key)
print('fields:', sorted(payload[first_key].keys()))
print('atom_count:', payload[first_key]['atom_count'])
print('xtb_valid:', payload[first_key]['xtb_valid'])
